# Notebook 2 — Two pipelines, measured end to end

Notebook 1 measures models one at a time. This one measures two complete ways of serving the
same twelve requests, so that the cost compared is the cost of a *policy* and not of a model.

- **Baseline** — every request goes straight to the large model. One call, no decision.
- **Cascade** — the smallest model answers first; a second, different model is asked whether
  that answer is acceptable; only if it says no does the request move up to a bigger model.

The cascade is the configuration an organization would actually deploy to save energy, so it
is measured with everything it costs in practice: the extra calls and the verifier itself.

This notebook does not depend on notebook 1. It only needs `prompt_bank.py` and Ollama.
Allow 30 to 45 minutes.

In [ ]:
import numpy as np
import pandas as pd

from core import ask, save_table
from prompt_bank import PROMPT_BANK

BASELINE = "qwen2.5:7b"                                        # the whole baseline pipeline
CHAIN = ["qwen2.5:0.5b", "qwen2.5:3b", "qwen2.5:7b"]           # cascade steps, smallest first
SCORER = "qwen2.5:1.5b"                                        # decides after each step

print(f"baseline   every request -> {BASELINE}")
print(f"cascade    {' -> '.join(CHAIN)}")
print(f"           with {SCORER} deciding after each step except the last")

## 1. How the verifier is asked, and why that matters

The verifier is asked for one word, and **only a clear OK accepts the answer**. Anything else
escalates: a refusal, an empty answer, a truncated one, an answer in the wrong format.

The opposite convention — looking for the word NON and accepting everything else — would make
an unreadable verifier output indistinguishable from an approval, and would quietly inflate the
energy saving. The whole result of this notebook turns on how often this one decision is wrong,
so the failure mode is chosen deliberately: when in doubt, escalate.

In [ ]:
SCORER_PROMPT = (
    "Tu vérifies une réponse. Si elle répond correctement et complètement à la question, "
    "réponds exactement OK. Sinon réponds exactement NON. Un seul mot.\n\n"
    "QUESTION :\n{question}\n\nRÉPONSE :\n{answer}"
)


def scorer_accepts(verdict_text):
    """True only for a clear OK. Everything else escalates."""
    return verdict_text.strip().upper().startswith("OK")

## 2. Run both pipelines

For each prompt, the baseline is measured first, then the cascade. Every call is measured
separately, and the cascade's total is the sum of everything it took to reach its final
answer — the intermediate attempts that were rejected included.

In [ ]:
def run_baseline(prompt):
    """One call to the large model."""
    result = ask(BASELINE, prompt["prompt"])
    return {
        "pipeline": "baseline", "prompt_id": prompt["id"], "tier": prompt["tier"],
        "final_model": BASELINE, "calls": 1, "escalations": 0,
        "energy_wh": result["energy_wh"],
        "energy_answers_wh": result["energy_wh"], "energy_scorer_wh": 0.0,
        "duration_s": result["duration_s"], "tokens_out": result["tokens_out"],
        "verdicts": "",
        "answer": result["answer"],
    }


def run_cascade(prompt):
    """Answer with the smallest model, verify, escalate only when the verifier refuses."""
    energy_answers = 0.0
    energy_scorer = 0.0
    duration = 0.0
    calls = 0
    escalations = 0
    verdicts = []

    for step, model in enumerate(CHAIN):
        attempt = ask(model, prompt["prompt"])
        calls += 1
        energy_answers += attempt["energy_wh"]
        duration += attempt["duration_s"]

        final_model = model
        final_answer = attempt["answer"]
        final_tokens = attempt["tokens_out"]

        is_last_step = step == len(CHAIN) - 1
        if is_last_step:
            break                       # nothing above this: keep this answer

        verdict = ask(SCORER,
                      SCORER_PROMPT.format(question=prompt["prompt"], answer=attempt["answer"]),
                      max_new_tokens=12, unload_first=False)
        calls += 1
        energy_scorer += verdict["energy_wh"]
        duration += verdict["duration_s"]
        verdicts.append(verdict["answer"].strip()[:20])

        if scorer_accepts(verdict["answer"]):
            break                       # accepted: keep this answer
        escalations += 1                # refused: try the next model up

    return {
        "pipeline": "cascade", "prompt_id": prompt["id"], "tier": prompt["tier"],
        "final_model": final_model, "calls": calls, "escalations": escalations,
        "energy_wh": energy_answers + energy_scorer,
        "energy_answers_wh": energy_answers, "energy_scorer_wh": energy_scorer,
        "duration_s": duration, "tokens_out": final_tokens,
        "verdicts": " | ".join(verdicts),
        "answer": final_answer,
    }


rows = []
for prompt in PROMPT_BANK:
    check = prompt["check"]
    for row in (run_baseline(prompt), run_cascade(prompt)):
        row["objective_check"] = check(row["answer"]) if check else np.nan
        rows.append(row)

    baseline_row, cascade_row = rows[-2], rows[-1]
    print(f"{prompt['id']:22} baseline {baseline_row['energy_wh']:7.4f} Wh   "
          f"cascade {cascade_row['energy_wh']:7.4f} Wh   "
          f"({cascade_row['calls']} calls, {cascade_row['escalations']} escalations, "
          f"ended on {cascade_row['final_model']})")

runs = save_table(pd.DataFrame(rows), "cascade_runs")

## 3. What the cascade costs and what it saves

In [ ]:
baseline = runs[runs.pipeline == "baseline"]
cascade = runs[runs.pipeline == "cascade"]

energy_baseline = baseline.energy_wh.mean()
energy_cascade = cascade.energy_wh.mean()
saving = 100 * (energy_cascade - energy_baseline) / energy_baseline

print(f"energy per request    {energy_baseline:.4f} -> {energy_cascade:.4f} Wh   {saving:+.1f} %")
print(f"latency per request   {baseline.duration_s.mean():6.1f} -> "
      f"{cascade.duration_s.mean():6.1f} s")
print(f"calls per request     {baseline.calls.mean():6.2f} -> {cascade.calls.mean():6.2f}")
print(f"\nthe verifier alone is "
      f"{100 * cascade.energy_scorer_wh.sum() / cascade.energy_wh.sum():.1f} %"
      f" of what the cascade consumes")

by_tier = runs.pivot_table(index="tier", columns="pipeline", values="energy_wh", aggfunc="mean")
by_tier["saving %"] = 100 * (by_tier.cascade - by_tier.baseline) / by_tier.baseline
print("\nmean energy per request, Wh")
print(by_tier.round(4).to_string())

## 4. Energy per request, then energy per *correct* answer

A saving on energy per request is not a saving if the answers are worse: the organization
simply pays again, in rework. The second figure divides the energy by the number of answers
that passed their deterministic check.

That second figure is computed on the **seven verifiable prompts only**. Dividing the energy of
all twelve by the correct answers among seven would mix two different populations and overstate
the result — which is exactly the kind of error this section exists to avoid.

In [ ]:
from scipy import stats

verifiable = runs[runs.objective_check.notna()]
v_baseline = verifiable[verifiable.pipeline == "baseline"]
v_cascade = verifiable[verifiable.pipeline == "cascade"]

correct_baseline = int(v_baseline.objective_check.sum())
correct_cascade = int(v_cascade.objective_check.sum())
n_verifiable = v_baseline.prompt_id.nunique()

print(f"objective accuracy    {correct_baseline}/{n_verifiable}  ->  "
      f"{correct_cascade}/{n_verifiable}")
print(f"energy per request    {saving:+.1f} %")

if correct_baseline > 0 and correct_cascade > 0:
    per_correct_baseline = v_baseline.energy_wh.sum() / correct_baseline
    per_correct_cascade = v_cascade.energy_wh.sum() / correct_cascade
    change = 100 * (per_correct_cascade - per_correct_baseline) / per_correct_baseline
    print(f"energy per correct    {change:+.1f} %   "
          f"({per_correct_baseline:.4f} -> {per_correct_cascade:.4f} Wh)")
else:
    print(f"energy per correct    not computable: one of the two pipelines answered none "
          f"of the {n_verifiable} verifiable prompts correctly")

contingency = [[correct_baseline, n_verifiable - correct_baseline],
               [correct_cascade, n_verifiable - correct_cascade]]
print(f"\nFisher exact test on the accuracy gap: p = {stats.fisher_exact(contingency)[1]:.3f}")
print("With seven prompts this test has very little power: it can confirm a large gap,")
print("not rule out a small one.")

## 5. How much does each figure depend on a single prompt?

Twelve prompts is few enough that one unusual request could carry the whole result. So each
figure is recomputed twelve times, leaving one prompt out each time, and the range is reported.

A range that stays on one side of zero means the direction of the result does not depend on any
single prompt. A range that crosses zero means the figure is a direction, not an estimate, and
should be written up as such.

In [ ]:
def leave_one_out(frame, per_correct_answer):
    """Recompute the saving once per prompt, each time without that prompt.

    Returns (lowest, highest), or None when the quantity cannot be computed."""
    results = []
    for prompt_id in sorted(frame.prompt_id.unique()):
        kept = frame[frame.prompt_id != prompt_id]
        left = kept[kept.pipeline == "baseline"]
        right = kept[kept.pipeline == "cascade"]

        if per_correct_answer:
            if left.objective_check.sum() == 0 or right.objective_check.sum() == 0:
                continue
            before = left.energy_wh.sum() / left.objective_check.sum()
            after = right.energy_wh.sum() / right.objective_check.sum()
        else:
            before, after = left.energy_wh.mean(), right.energy_wh.mean()

        results.append(100 * (after - before) / before)

    return (min(results), max(results)) if results else None


for label, span in [("per request", leave_one_out(runs, False)),
                    ("per correct answer", leave_one_out(verifiable, True))]:
    if span is None:
        print(f"{label:20} not computable on this campaign")
        continue
    low, high = span
    verdict = "never changes sign" if low * high > 0 else "CHANGES SIGN — report as a direction"
    print(f"{label:20} {low:+7.1f} % to {high:+7.1f} %   "
          f"({high - low:5.1f} points wide)   {verdict}")

## 6. Where the verifier was wrong

Two failure modes, and only one of them is visible in the energy figures. Escalating an answer
that was already good costs energy. Accepting an answer that was wrong costs nothing and looks
like a saving — which is why it has to be counted explicitly.

In [ ]:
print("where each request ended up")
print(cascade.final_model.value_counts().to_string())
print(f"\n{int(cascade.escalations.sum())} escalations over {len(cascade)} requests")

approved_but_wrong = cascade[(cascade.objective_check == 0) & (cascade.escalations == 0)]
checkable_cascade = cascade[cascade.objective_check.notna()]
print(f"\nwrong answers the verifier accepted without escalating: "
      f"{len(approved_but_wrong)} of {len(checkable_cascade)} verifiable requests")
if len(approved_but_wrong):
    print(approved_but_wrong[["prompt_id", "tier", "final_model", "verdicts"]]
          .to_string(index=False))

## 7. Figures

Blue is the baseline, orange the cascade — the same pair throughout, and one that stays
readable in greyscale and for colour-blind readers.

In [ ]:
from core import (BASELINE_COLOUR, CASCADE_COLOUR, TIER_LABELS,
                  new_figure, save_figure)

tiers = sorted(runs.tier.unique())
positions = np.arange(len(tiers))
labels = [TIER_LABELS.get(tier, tier) for tier in tiers]

figure, axes = new_figure(
    "Energy per request: always the large model, against the cascade",
    "Mean over the prompts of each tier. The cascade total includes the rejected attempts "
    "and the verifier.",
    "Energy per request (Wh)", width=10)

for index, (pipeline, colour) in enumerate([("baseline", BASELINE_COLOUR),
                                            ("cascade", CASCADE_COLOUR)]):
    heights = [runs[(runs.pipeline == pipeline) & (runs.tier == tier)].energy_wh.mean()
               for tier in tiers]
    axes.bar(positions + index * 0.38 - 0.19, heights, 0.34, label=pipeline,
             color=colour, zorder=3)

axes.set_xticks(positions)
axes.set_xticklabels(labels)
save_figure(figure, axes, "F5_cascade_energy", legend_columns=2)


answers = [cascade[cascade.tier == tier].energy_answers_wh.mean() for tier in tiers]
scorer = [cascade[cascade.tier == tier].energy_scorer_wh.mean() for tier in tiers]

figure, axes = new_figure(
    "Where the cascade spends its energy",
    "Producing answers, including the attempts that were rejected, against deciding "
    "whether they were good enough.",
    "Energy per request (Wh)", width=10)
axes.bar(positions, answers, 0.42, label="producing the answers",
         color=BASELINE_COLOUR, zorder=3)
axes.bar(positions, scorer, 0.42, bottom=answers, label="verifying them",
         color=CASCADE_COLOUR, zorder=3)
axes.set_xticks(positions)
axes.set_xticklabels(labels)
save_figure(figure, axes, "F6_cascade_split", legend_columns=2)

## 8. Output of this notebook

In `outputs/`:

| File | What it holds |
|---|---|
| `cascade_runs.csv` | one row per pipeline and prompt: 24 rows, with the verdicts and the final answers |
| `figures/F5_cascade_energy.png` | baseline against cascade, by tier |
| `figures/F6_cascade_split.png` | answering against verifying |